# Analise arquivo raw clientes.csv

In [158]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [159]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR =  os.path.join(DATA_DIR, 'raw')

In [160]:
df = pd.read_csv(os.path.join(RAW_DIR, 'clientes.csv'), sep=';')
df.shape


(51, 7)

## Analise exploratória

In [161]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id_cliente      51 non-null     str  
 1   nome_cliente    50 non-null     str  
 2   cidade          50 non-null     str  
 3   uf              50 non-null     str  
 4   segmento        50 non-null     str  
 5   limite_credito  49 non-null     str  
 6   status          50 non-null     str  
dtypes: str(7)
memory usage: 2.9 KB


In [162]:
df.isna().sum()

id_cliente        0
nome_cliente      1
cidade            1
uf                1
segmento          1
limite_credito    2
status            1
dtype: int64

In [163]:
df.head()

,id_cliente,nome_cliente,cidade,uf,segmento,limite_credito,status
0,C0001,Comercial Presidente 01,Regente Feijó,SP,Varejo,60616.29,Ativo
1,C0002,Mercado Central 02,NaN,SP,Varejo,55752.46,Ativo
2,C0003,Supermercado Avenida 03,Santo Anastácio,SP,Varejo,7234.79,Ativo
3,C0004,Mercado Central 04,Presidente Prudente,SP,Mercado,58701.47,Ativo
4,C0005,Supermercado União 05,pirapozinho,SP,Food Service,49194.93,Ativo


In [164]:
df.tail()

,id_cliente,nome_cliente,cidade,uf,segmento,limite_credito,status
46,C0047,Supermercado Avenida 47,Presidente Prudente,SP,Varejo,77674.72,Ativo
47,C0048,Comercial Oeste 48,Santo Anastácio,SP,Food Service,41101.88,Ativo
48,C0049,Mercado Esperança 49,Presidente Prudente,SP,Mercado,33422.98,Ativo
49,C0050,Empório Prudente 50,Rancharia,NaN,Atacado,36725.56,Ativo
50,C0014,Mercado Ideal 14,Regente Feijó,SP,Mercado,NaN,Ativo


In [165]:
df.sample(5)

,id_cliente,nome_cliente,cidade,uf,segmento,limite_credito,status
0,C0001,Comercial Presidente 01,Regente Feijó,SP,Varejo,60616.29,Ativo
19,C0020,Mercado Ideal 20,Martinópolis,SP,Food Service,72336.72,Ativo
37,C0038,Distribuidora Nova Era 38,Presidente Bernardes,SP,Varejo,43561.71,Ativo
24,C0025,Comercial Presidente 25,Martinópolis,SP,Atacado,13366.41,Ativo
49,C0050,Empório Prudente 50,Rancharia,NaN,Atacado,36725.56,Ativo


## Tratamento de dados

In [166]:
df_original = df.copy()

In [167]:
df[['limite_credito']] = df[['limite_credito']].fillna(0)

In [168]:
df.isnull().sum()

id_cliente        0
nome_cliente      1
cidade            1
uf                1
segmento          1
limite_credito    0
status            1
dtype: int64

In [169]:
q = '''SELECT id_cliente
                , nome_cliente
                , cidade
                , uf
                , segmento
                , status
            FROM df
        WHERE nome_cliente isnull
                or cidade isnull
                or uf isnull
                or segmento isnull
                or status isnull
    '''
print(pysqldf(q))

  id_cliente         nome_cliente                cidade   uf segmento status
0      C0002   Mercado Central 02                   NaN   SP   Varejo  Ativo
1      C0030                  NaN  Presidente Bernardes   SP  Atacado  Ativo
2      C0036    Rede Bom Preço 36   Presidente Prudente   SP      NaN    NaN
3      C0050  Empório Prudente 50             Rancharia  NaN  Atacado  Ativo


In [170]:
df = df.dropna()

In [171]:
df.info()

<class 'pandas.DataFrame'>
Index: 47 entries, 0 to 50
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_cliente      47 non-null     str   
 1   nome_cliente    47 non-null     str   
 2   cidade          47 non-null     str   
 3   uf              47 non-null     str   
 4   segmento        47 non-null     str   
 5   limite_credito  47 non-null     object
 6   status          47 non-null     str   
dtypes: object(1), str(6)
memory usage: 2.9+ KB


In [181]:
df['limite_credito'] = pd.to_numeric(df['limite_credito'], errors='coerce')

In [183]:
df.info()

<class 'pandas.DataFrame'>
Index: 47 entries, 0 to 50
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_cliente      47 non-null     str    
 1   nome_cliente    47 non-null     str    
 2   cidade          47 non-null     str    
 3   uf              47 non-null     str    
 4   segmento        47 non-null     str    
 5   limite_credito  46 non-null     float64
 6   status          47 non-null     str    
dtypes: float64(1), str(6)
memory usage: 2.9 KB


## Salvando em Bronze

In [184]:
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')

In [185]:
df.to_csv(os.path.join(BRONZE_DIR, 'b_clientes.csv'), sep=',')